## IMPLEMENTING A GPT MODEL FROM SCRATCH TO GENERATE TEXT

In [1]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of transformer blocks
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

## GPT Architecture Part1: Dummy GPT Model Class
<div class="alert alert-block alert-info">

Step 1: Use a placeholder for TransformerBlock

Step 2: Use a placeholder for LayerNorm
</div

In [2]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential( *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)


    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # Placeholder implementation
    def forward(self, x):
        # This block does nothing and just returns its input.
        return x
class DummyLayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        # Placeholder implementation
    def forward(self, x):
        # This layer norm does nothing and just returns its input.
        return x

    



<div class="alert alert-block alert-warning">

The DummyGPTModel class in this code defines a simplified version of a GPT-like model using
PyTorch's neural network module (nn.Module). 

The model architecture in the
DummyGPTModel class consists of token and positional embeddings, dropout, a series of
transformer blocks (DummyTransformerBlock), a final layer normalization
(DummyLayerNorm), and a linear output layer (out_head). 

The configuration is passed in via
a Python dictionary, for instance, the GPT_CONFIG_124M dictionary we created earlier.

</div>

<div class="alert alert-block alert-warning">
    
The forward method describes the data flow through the model: it computes token and
positional embeddings for the input indices, applies dropout, processes the data through
the transformer blocks, applies normalization, and finally produces logits with the linear
output layer.

</div>

<div class="alert alert-block alert-success">

Next, we will prepare the input data and initialize a new GPT model to illustrate its
usage.

</div>

## Step1: TOKENTIZATION

In [3]:
import torch
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)  # Shape: (batch_size, seq_length)
print("Input batch shape:", batch.shape) # 2 batches with 4 tokens each

Input batch shape: torch.Size([2, 4])


## Step2: Create an instance of the DummyModel

In [5]:
import torch
torch.manual_seed(42)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("Output logits shape:", logits.shape)  # Should be (batch_size, seq_length, vocab_size)
print(logits)


Output logits shape: torch.Size([2, 4, 50257])
tensor([[[ 0.1597,  0.1958, -0.1044,  ...,  0.0555,  0.4726, -0.5874],
         [-0.7750,  0.5466, -1.2552,  ..., -0.2906,  0.1688,  0.4970],
         [-0.9552, -1.1343,  0.4898,  ..., -0.0489, -0.2273,  0.1706],
         [ 1.3697, -0.8169, -0.2184,  ...,  0.9854,  0.4593,  0.6592]],

        [[ 0.8104, -0.2729, -0.0559,  ...,  0.3231,  0.6338, -1.0197],
         [-0.5044,  0.9103, -1.0994,  ..., -0.4777,  0.5505, -0.5228],
         [-0.5149,  0.4777,  1.1838,  ..., -1.0277, -0.6529, -0.0847],
         [ 0.9222,  0.3893, -0.4700,  ...,  0.3818,  0.5648,  1.9670]]],
       grad_fn=<UnsafeViewBackward0>)


<div class="alert alert-block alert-warning">

The output tensor has two rows corresponding to the two text samples. Each text sample
consists of 4 tokens; each token is a 50,257-dimensional vector, which matches the size of
the tokenizer's vocabulary.


The embedding has 50,257 dimensions because each of these dimensions refers to a
unique token in the vocabulary. At the end of this chapter, when we implement the
postprocessing code, we will convert these 50,257-dimensional vectors back into token IDs,
which we can then decode into words.

</div>

<div class="alert alert-block alert-warning">

Now that we have taken a top-down look at the GPT architecture and its in- and outputs,
we will code the individual placeholders in the upcoming sections, starting with the real
layer normalization class that will replace the DummyLayerNorm in the previous code.
</div>